# Drishti: fine-tune Prithvi-EO-2.0 for dumpsite detection (B-08, B-09)
Run on a Colab GPU runtime (Runtime > Change runtime type > GPU). Needs the labelled GeoJSON for the pilot city in `pipeline/labels/`.

In [ ]:
!git clone https://github.com/saxenachirayu1/drishti.git
%cd drishti/pipeline
!pip install -q -e ".[geo,train]"

## 1. Check the labels (B-06)

In [ ]:
!drishti-pipeline labels-check labels/pune.geojson

## 2. Build the training tiles from the clearest dry-season scenes (B-07)

In [ ]:
!drishti-pipeline build-dataset --city Pune --start 2026-01-01 --end 2026-04-30 --scenes 3
!find data/dataset -name '*_img.tif' | cut -d/ -f3 | sort | uniq -c

## 3. Fine-tune (B-08)
Confirm the means/stds in `training/prithvi_dumpsite.yaml` against the checkpoint's model card first.

In [ ]:
!terratorch fit -c training/prithvi_dumpsite.yaml

## 4. Test-set metrics (B-09)

In [ ]:
import glob
ckpt = sorted(glob.glob('outputs/prithvi_dumpsite/**/best*.ckpt', recursive=True))[-1]
print(ckpt)
!terratorch test -c training/prithvi_dumpsite.yaml --ckpt_path {ckpt}

## 5. Compare with the spectral baseline (B-10)
Run detection with and without the checkpoint over the test area, export both as GeoJSON, then:

```
drishti-pipeline evaluate --pred baseline.geojson --truth labels/pune.geojson
drishti-pipeline evaluate --pred prithvi.geojson --truth labels/pune.geojson
```
Copy the two lines into `training/RESULTS.md` and the pitch.

## 6. Save the checkpoint
Download `best.ckpt` (it is not committed: see `.gitignore`) and upload it where the worker can read it, then set `DRISHTI_CHECKPOINT`.